# 🔬 Continual Anomaly Detection (VisA) - Google Colab

Questo notebook permette di configurare ed eseguire il progetto **Continual Anomaly Detection** su Google Colab, utilizzando il dataset **VisA**.

### Prerequisiti
- Seleziona un runtime con **GPU** (Runtime → Cambia tipo di runtime → T4 GPU)
- Il dataset VisA deve essere caricato su **Google Drive**

## 0. Verifica GPU
Controlliamo che il runtime abbia una GPU disponibile.

In [ ]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print(f"✅ GPU disponibile: {gpu_name}")
    print(f"   Memoria totale: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("⚠️ ATTENZIONE: Nessuna GPU rilevata!")
    print("   Vai su Runtime → Cambia tipo di runtime → T4 GPU")

## 1. Clonare la Repository
Clona il repository da GitHub ed entra nella directory del progetto.

Se il repository è **privato**, imposta il token nella variabile `GITHUB_TOKEN`.

In [ ]:
import os

REPO_URL = "https://github.com/roccopiovardaro/Continual_Anomaly_Detection.git"
REPO_DIR = "Continual_Anomaly_Detection"

# Se il repository è privato, inserisci il tuo GitHub Personal Access Token qui:
GITHUB_TOKEN = ""  # lascia vuoto se il repo è pubblico

if not os.path.exists(REPO_DIR):
    if GITHUB_TOKEN:
        clone_url = REPO_URL.replace("https://", f"https://{GITHUB_TOKEN}@")
    else:
        clone_url = REPO_URL
    !git clone {clone_url}
    print(f"✅ Repository clonato in {REPO_DIR}")
else:
    print(f"ℹ️ La directory {REPO_DIR} esiste già, skip clone.")
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
print(f"📂 Working directory: {os.getcwd()}")

## 2. Installare le Dipendenze
Installiamo le librerie necessarie. PyTorch e torchvision sono già preinstallati su Colab con supporto CUDA, quindi li escludiamo per evitare conflitti.

In [ ]:
!pip install efficientnet_pytorch einops imgaug timm scipy scikit-learn \
    Pillow PyYAML tqdm pandas matplotlib codecarbon umap-learn opencv-python

print("\n✅ Dipendenze installate!")
print(f"   PyTorch: {torch.__version__}")
print(f"   CUDA disponibile: {torch.cuda.is_available()}")

## 3. Scaricare il Modello Preaddestrato (solo se usi ViT)
Se il config usa `model.name: vit`, il checkpoint ViT-B/16 verrà scaricato nella cartella `checkpoints/`.

**Se usi `convnext` o `resnet`, puoi saltare questa cella.**

In [ ]:
import os

os.makedirs("checkpoints", exist_ok=True)

vit_path = "checkpoints/ViT-B_16.npz"
if not os.path.exists(vit_path):
    print("⬇️ Downloading ViT-B/16 checkpoint...")
    !wget -q --show-progress https://storage.googleapis.com/vit_models/sam/ViT-B_16.npz -P checkpoints/
    print("✅ Checkpoint scaricato!")
else:
    print("ℹ️ Checkpoint ViT-B/16 già presente, skip download.")

## 4. Collegare Google Drive
Montiamo Google Drive per accedere al dataset VisA.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive montato in /content/drive")

## 6. Configurazione dell'Esperimento

I parametri dell'esperimento sono definiti nel file `configs/cad.yaml`.

**Modifica il file YAML direttamente** prima di eseguire il training.

In [ ]:
# ============================================================
# CONFIGURAZIONE ESPERIMENTO
# ============================================================
CONFIG_FILE = "./configs/cad.yaml"
DEVICE = "cuda"   # 'cuda' su Colab (default), 'cpu' come fallback
SEED   = 42

# Mostra il contenuto del config YAML
print("📄 Contenuto di", CONFIG_FILE)
print("=" * 50)
with open(CONFIG_FILE, 'r') as f:
    print(f.read())

print("📋 Riepilogo:")
print(f"   Config file: {CONFIG_FILE}")
print(f"   Device:      {DEVICE}")
print(f"   Seed:        {SEED}")
print(f"   Data dir:    {DATA_DIR}")

## 7. Eseguire il Training 🚀

Esegui `main.py` con i parametri configurati sopra.

Il training produce:
- **Checkpoint** del modello in `checkpoints/`
- **Statistiche** dei tempi in `training_times.csv`
- **Visualizzazioni** (istogrammi, t-SNE, UMAP) nelle rispettive cartelle

In [ ]:
import yaml

# ============================================================
# CONFIGURAZIONE ESPERIMENTO
# ============================================================
DEVICE = "cuda"   # 'cuda' su Colab (default), 'cpu' come fallback
SEED   = 42

# Configurazione YAML per VisA
config = {
    'name': 'continual anomaly detection - VisA',
    'dataset': {
        'name': 'seq-visa',
        'image_size': 224,
        'num_workers': 2,         # Colab ha risorse limitate, 2 è un buon compromesso
        'data_incre_setting': 'mul',
        'n_classes_per_task': 3,   # 12 classi VisA / 4 task = 3 classi per task
        'n_tasks': 4,             # VisA con setting 'mul': 4 task
        'dataset_order': 1,       # ordine delle classi (1, 2, o 3)
        'strong_augmentation': True,
        'random_aug': False,
    },
    'model': {
        'name': 'convnext',
        'pretrained': True,
        'method': 'dne_replay_ewc',
        'fix_head': True,
        'with_dne': True,
        'with_embeds': True,
        'buffer_size': 200,
        'n_feat': 304,
        'fc_internal': 1024,
        'n_coupling_blocks': 4,
        'clamp': 3,
        'n_scales': 3,
    },
    'train': {
        'optimizer': {
            'name': 'adam',
            'weight_decay': 0.00003,
            'momentum': 0.9,
        },
        'warmup_epochs': 10,
        'warmup_lr': 0,
        'base_lr': 0.0001,
        'final_lr': 0,
        'num_epochs': 20,
        'batch_size': 8,
        'test_epochs': 5,
        'alpha': 0.4,
        'beta': 0.5,
        'num_classes': 2,
        'ewc_lambda': 5000.0,
        'real_embed_ratio': 0.05,
    },
    'eval': {
        'eval_classifier': 'density',
        'batch_size': 16,
        'visualization': True,
    },
}

# Salva il config YAML
CONFIG_FILE = "./configs/cad_visa.yaml"
with open(CONFIG_FILE, 'w') as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

# Stampa la configurazione
print("📋 Configurazione:")
print(f"   Config file: {CONFIG_FILE}")
print(f"   Dataset:     seq-visa (12 classi, 4 task x 3 classi)")
print(f"   Modello:     {config['model']['name']} ({config['model']['method']})")
print(f"   Epoche:      {config['train']['num_epochs']}")
print(f"   Batch size:  {config['train']['batch_size']}")
print(f"   Device:      {DEVICE}")
print(f"   Seed:        {SEED}")
print(f"   Data dir:    {DATA_DIR}")

print("\n📄 Config YAML generato:")
print("=" * 50)
with open(CONFIG_FILE, 'r') as f:
    print(f.read())

## 7. Eseguire il Training 🚀

Esegui `main.py` con i parametri configurati sopra.

Il training su VisA produce:
- **4 task** sequenziali, ciascuno con 3 classi
- **Checkpoint** del modello in `checkpoints/`
- **Statistiche** dei tempi in `training_times.csv`
- **Visualizzazioni** (istogrammi, t-SNE, UMAP) nelle rispettive cartelle

In [ ]:
!python main.py \
    --config-file {CONFIG_FILE} \
    --data_dir "{DATA_DIR}" \
    --device {DEVICE} \
    --seed {SEED}

## 8. Risultati e Visualizzazione
Dopo il training, visualizziamo i risultati prodotti.

In [ ]:
import pandas as pd
import os

# Mostra i tempi di training
csv_path = "training_times.csv"
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print("⏱️ Statistiche sui tempi di training:")
    display(df)
    
    # Mostra il tempo totale per task
    task_totals = df[df['Time_Type'] == 'Task_Train_Total']
    if not task_totals.empty:
        print("\n📊 Tempo totale per task:")
        for _, row in task_totals.iterrows():
            mins = row['Duration_Seconds'] / 60
            print(f"   Task {int(row['Task'])}: {row['Duration_Seconds']:.1f}s ({mins:.1f} min)")
else:
    print("⚠️ File training_times.csv non trovato. Esegui prima il training.")

In [ ]:
import glob
from IPython.display import Image, display

# Mostra le visualizzazioni generate (istogrammi, t-SNE, UMAP)
for folder, label in [("hist_results", "📊 Istogrammi"), 
                       ("tsne_results", "🔵 t-SNE"), 
                       ("umap_results", "🟣 UMAP")]:
    images = sorted(glob.glob(f"{folder}/**/*.png", recursive=True))
    if images:
        print(f"\n{label} ({len(images)} immagini):")
        for img_path in images[-6:]:  # mostra le ultime 6
            print(f"  → {img_path}")
            display(Image(filename=img_path, width=600))
    else:
        print(f"\n{label}: nessuna immagine trovata in {folder}/")

## 9. Salvare i Risultati su Google Drive (Opzionale)
Copia checkpoint e risultati sul tuo Google Drive per conservarli.

In [ ]:
import shutil
import os

# ⚠️ Modifica questo percorso con la destinazione su Google Drive
SAVE_DIR = "/content/drive/MyDrive/CAD_results_visa"

os.makedirs(SAVE_DIR, exist_ok=True)

# Copia i risultati
for folder in ["checkpoints", "hist_results", "tsne_results", "umap_results"]:
    if os.path.exists(folder):
        dest = os.path.join(SAVE_DIR, folder)
        if os.path.exists(dest):
            shutil.rmtree(dest)
        shutil.copytree(folder, dest)
        print(f"✅ {folder} → {dest}")

# Copia il CSV dei tempi
if os.path.exists("training_times.csv"):
    shutil.copy2("training_times.csv", SAVE_DIR)
    print(f"✅ training_times.csv → {SAVE_DIR}")

# Copia il config usato
shutil.copy2(CONFIG_FILE, SAVE_DIR)
print(f"✅ {CONFIG_FILE} → {SAVE_DIR}")

print(f"\n🎉 Risultati salvati in {SAVE_DIR}")